In [2]:
!pip install optuna
import optuna
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 17.8 MB/s eta 0:00:00


In [3]:

# Load sample dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
feature_names = X.columns.tolist()

def objective(trial):
    # --- FEATURE SELECTION: pick which features to include ---
    selected = [f for f in feature_names
                if trial.suggest_categorical(f, [True, False])]

    if len(selected) == 0:
        return 0.0  # skip empty subsets

    X_subset = X[selected]

    # --- HYPERPARAMETER TUNING: model settings ---
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 20),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 10),
        random_state=42
    )

    score = cross_val_score(model, X_subset, y, cv=5, scoring='accuracy').mean()
    return score



In [4]:
# Run the search
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10, show_progress_bar=True)

# ===== RESULTS =====


[I 2026-07-19 11:26:22,440] A new study created in memory with name: no-name-99411d15-8c07-4eca-af01-64cd5aa079db


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-07-19 11:26:26,402] Trial 0 finished with value: 0.9068467629250117 and parameters: {'mean radius': False, 'mean texture': True, 'mean perimeter': False, 'mean area': False, 'mean smoothness': True, 'mean compactness': False, 'mean concavity': False, 'mean concave points': False, 'mean symmetry': False, 'mean fractal dimension': True, 'radius error': False, 'texture error': False, 'perimeter error': False, 'area error': False, 'smoothness error': True, 'compactness error': False, 'concavity error': True, 'concave points error': False, 'symmetry error': False, 'fractal dimension error': True, 'worst radius': False, 'worst texture': False, 'worst perimeter': False, 'worst area': False, 'worst smoothness': False, 'worst compactness': True, 'worst concavity': True, 'worst concave points': False, 'worst symmetry': True, 'worst fractal dimension': True, 'n_estimators': 153, 'max_depth': 20, 'min_samples_split': 6}. Best is trial 0 with value: 0.9068467629250117.
[I 2026-07-19 11:26:2

In [5]:
X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [6]:
# 1. Best accuracy found
print(f"Best CV Accuracy: {study.best_value:.4f}")

# 2. Best feature combination
best_features = [f for f in feature_names if study.best_params[f]]
print(f"\nSelected Features ({len(best_features)}):")
print(best_features)


Best CV Accuracy: 0.9631

Selected Features (22):
['mean radius', 'mean perimeter', 'mean area', 'mean smoothness', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'area error', 'compactness error', 'concavity error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concave points', 'worst symmetry', 'worst fractal dimension']


In [7]:

# 3. Best hyperparameters
hyperparam_names = ['n_estimators', 'max_depth', 'min_samples_split']
best_hyperparams = {k: v for k, v in study.best_params.items() if k in hyperparam_names}
print(f"\nBest Hyperparameters:")
print(best_hyperparams)

# 4a. Importance of the SELECTED FEATURES ONLY
# (recomputes ANOVA decomposition restricted to feature params)
feature_importance = optuna.importance.get_param_importances(
    study, params=best_features
)
print(f"\nFeature Importance (only selected features):")
for param, score in feature_importance.items():
    print(f" {param}: {score:.4f}")



Best Hyperparameters:
{'n_estimators': 197, 'max_depth': 13, 'min_samples_split': 2}

Feature Importance (only selected features):
 symmetry error: 0.2812
 mean fractal dimension: 0.1299
 area error: 0.1186
 worst texture: 0.0864
 worst concave points: 0.0751
 radius error: 0.0608
 mean radius: 0.0421
 worst area: 0.0377
 mean area: 0.0336
 worst symmetry: 0.0326
 concavity error: 0.0278
 worst smoothness: 0.0249
 worst fractal dimension: 0.0236
 compactness error: 0.0084
 mean perimeter: 0.0056
 mean smoothness: 0.0047
 mean symmetry: 0.0034
 worst compactness: 0.0016
 worst radius: 0.0012
 fractal dimension error: 0.0004
 texture error: 0.0004
 worst perimeter: 0.0001


In [9]:

# 4b. Importance of the HYPERPARAMETERS ONLY
# (recomputes ANOVA decomposition restricted to hyperparameter params)
hyperparam_importance = optuna.importance.get_param_importances(
    study, params=hyperparam_names
)
print(f"\nHyperparameter Importance (only n_estimators, max_depth, min_samples_split):")
for param, score in hyperparam_importance.items():
    print(f" {param}: {score:.4f}")


Hyperparameter Importance (only n_estimators, max_depth, min_samples_split):
 max_depth: 0.8054
 n_estimators: 0.1890
 min_samples_split: 0.0056


In [11]:
import optuna
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Load sample dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
feature_names = X.columns.tolist()

def objective(trial):
    # --- FEATURE SELECTION: pick which features to include ---
    selected = [f for f in feature_names
                if trial.suggest_categorical(f, [True, False])]

    if len(selected) == 0:
        return 0.0  # skip empty subsets

    X_subset = X[selected]

    # --- HYPERPARAMETER TUNING: model settings ---
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 20),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 10),
        random_state=42
    )

    score = cross_val_score(model, X_subset, y, cv=5, scoring='accuracy').mean()
    return score

# Run the search
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10, show_progress_bar=True)

# ===== RESULTS =====

# 1. Best accuracy found
print(f"Best CV Accuracy: {study.best_value:.4f}")

# 2. Best feature combination
best_features = [f for f in feature_names if study.best_params[f]]
print(f"\nSelected Features ({len(best_features)}):")
print(best_features)

# 3. Best hyperparameters
hyperparam_names = ['n_estimators', 'max_depth', 'min_samples_split']
best_hyperparams = {k: v for k, v in study.best_params.items() if k in hyperparam_names}
print(f"\nBest Hyperparameters:")
print(best_hyperparams)

# 4a. Importance of the SELECTED FEATURES ONLY
# (recomputes ANOVA decomposition restricted to feature params)
feature_importance = optuna.importance.get_param_importances(
    study, params=best_features
)
print(f"\nFeature Importance (only selected features):")
for param, score in feature_importance.items():
    print(f" {param}: {score:.4f}")

# 4b. Importance of the HYPERPARAMETERS ONLY
# (recomputes ANOVA decomposition restricted to hyperparameter params)
hyperparam_importance = optuna.importance.get_param_importances(
    study, params=hyperparam_names
)
print(f"\nHyperparameter Importance (only n_estimators, max_depth, min_samples_split):")
for param, score in hyperparam_importance.items():
    print(f" {param}: {score:.4f}")

[I 2026-07-19 10:57:57,880] A new study created in memory with name: no-name-7bd382b6-22ab-4804-a395-53c39446b849


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-07-19 10:58:00,815] Trial 0 finished with value: 0.9595870206489675 and parameters: {'mean radius': True, 'mean texture': True, 'mean perimeter': False, 'mean area': False, 'mean smoothness': True, 'mean compactness': True, 'mean concavity': True, 'mean concave points': False, 'mean symmetry': True, 'mean fractal dimension': False, 'radius error': True, 'texture error': False, 'perimeter error': True, 'area error': True, 'smoothness error': False, 'compactness error': False, 'concavity error': False, 'concave points error': True, 'symmetry error': False, 'fractal dimension error': True, 'worst radius': True, 'worst texture': True, 'worst perimeter': True, 'worst area': True, 'worst smoothness': True, 'worst compactness': True, 'worst concavity': False, 'worst concave points': False, 'worst symmetry': False, 'worst fractal dimension': False, 'n_estimators': 261, 'max_depth': 11, 'min_samples_split': 8}. Best is trial 0 with value: 0.9595870206489675.
[I 2026-07-19 10:58:03,949] 

In [1]:
X


NameError: name 'X' is not defined

In [12]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [17]:
# 1. Optimization History
plot_optimization_history(study).show()

In [18]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

AttributeError: 'tuple' object has no attribute '_is_multi_objective'

In [19]:
# 3. Slice Plot
plot_slice(study).show()

In [20]:
# 4. Contour Plot
plot_contour(study).show()

In [16]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [29]:
from optuna.visualization import plot_param_importances

# Feature importance plot
fig1 = plot_param_importances(study, params=feature_names)
fig1.update_layout(title="Feature Importance")
fig1.show()

# Hyperparameter importance plot
fig2 = plot_param_importances(study, params=hyperparam_names)
fig2.update_layout(title="Hyperparameter Importance")
fig2.show()

In [28]:
import optuna
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

# Load sample dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
feature_names = X.columns.tolist()

def objective(trial):
    # --- FEATURE SELECTION: pick which features to include ---
    selected = [f for f in feature_names
                if trial.suggest_categorical(f, [True, False])]

    if len(selected) == 0:
        return 0.0  # skip empty subsets

    X_subset = X[selected]

    # --- HYPERPARAMETER TUNING: model settings ---
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 20),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 10),
        random_state=42
    )

    # Manual CV loop so we can report intermediate scores per fold —
    # required for the pruner to have anything to compare against.
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []

    for step, (train_idx, val_idx) in enumerate(cv.split(X_subset, y)):
        X_train, X_val = X_subset.iloc[train_idx], X_subset.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model.fit(X_train, y_train)
        fold_acc = model.score(X_val, y_val)
        fold_scores.append(fold_acc)

        # Report intermediate result and check if this trial should be pruned
        trial.report(np.mean(fold_scores), step)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(fold_scores)

# Single study, with a pruner that can now actually act on reported values
study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)  # don't prune before fold 2
)
study.optimize(objective, n_trials=20, show_progress_bar=True)

# Check how many trials got pruned vs completed
pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
completed = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
print(f"Completed: {completed}, Pruned early: {pruned}")

# ===== RESULTS =====

# 1. Best accuracy found
print(f"Best CV Accuracy: {study.best_value:.4f}")

# 2. Best feature combination
best_features = [f for f in feature_names if study.best_params[f]]
print(f"\nSelected Features ({len(best_features)}):")
print(best_features)

# 3. Best hyperparameters
hyperparam_names = ['n_estimators', 'max_depth', 'min_samples_split']
best_hyperparams = {k: v for k, v in study.best_params.items() if k in hyperparam_names}
print(f"\nBest Hyperparameters:")
print(best_hyperparams)

# 4a. Importance of features only
feature_importance = optuna.importance.get_param_importances(
    study, params=feature_names
)
print(f"\nFeature Importance (weightage):")
for param, score in feature_importance.items():
    print(f" {param}: {score:.4f}")

# 4b. Importance of hyperparameters only
hyperparam_importance = optuna.importance.get_param_importances(
    study, params=hyperparam_names
)
print(f"\nHyperparameter Importance (weightage):")
for param, score in hyperparam_importance.items():
    print(f" {param}: {score:.4f}")

[I 2026-07-19 11:49:38,060] A new study created in memory with name: no-name-a31ce2ae-1a8c-4b1d-bc9b-f6c8deb62dc9


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-07-19 11:49:42,944] Trial 0 finished with value: 0.952585002328831 and parameters: {'mean radius': True, 'mean texture': True, 'mean perimeter': True, 'mean area': True, 'mean smoothness': True, 'mean compactness': False, 'mean concavity': False, 'mean concave points': True, 'mean symmetry': True, 'mean fractal dimension': True, 'radius error': True, 'texture error': False, 'perimeter error': True, 'area error': False, 'smoothness error': False, 'compactness error': False, 'concavity error': True, 'concave points error': True, 'symmetry error': False, 'fractal dimension error': True, 'worst radius': True, 'worst texture': False, 'worst perimeter': True, 'worst area': False, 'worst smoothness': True, 'worst compactness': True, 'worst concavity': False, 'worst concave points': True, 'worst symmetry': True, 'worst fractal dimension': True, 'n_estimators': 196, 'max_depth': 10, 'min_samples_split': 4}. Best is trial 0 with value: 0.952585002328831.
[I 2026-07-19 11:49:44,507] Trial

# **For MLflow+**

In [ ]:
import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.data
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

# ===== SETUP =====
mlflow.set_experiment("breast_cancer_feature_hparam_search")

# Load sample dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
feature_names = X.columns.tolist()

# Build a combined dataframe + MLflow dataset object once, so every run can log it
full_df = X.copy()
full_df['target'] = y
mlflow_dataset = mlflow.data.from_pandas(
    full_df, source="sklearn.datasets.load_breast_cancer", name="breast_cancer_dataset"
)

def objective(trial):
    # --- FEATURE SELECTION: pick which features to include ---
    selected = [f for f in feature_names
                if trial.suggest_categorical(f, [True, False])]

    if len(selected) == 0:
        return 0.0  # skip empty subsets

    X_subset = X[selected]

    # --- HYPERPARAMETER TUNING: model settings ---
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )

    # Manual CV loop so we can report intermediate scores per fold (needed for pruning)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []

    for step, (train_idx, val_idx) in enumerate(cv.split(X_subset, y)):
        X_train, X_val = X_subset.iloc[train_idx], X_subset.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model.fit(X_train, y_train)
        fold_acc = model.score(X_val, y_val)
        fold_scores.append(fold_acc)

        trial.report(np.mean(fold_scores), step)
        if trial.should_prune():
            raise optuna.TrialPruned()

    mean_acc = np.mean(fold_scores)

    # --- MLFLOW: log this trial as a nested run ---
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):
        mlflow.log_params({
            'n_estimators': n_estimators,
            'max_depth': max_depth,
            'min_samples_split': min_samples_split,
            'n_features_selected': len(selected)
        })
        mlflow.log_metric('cv_accuracy', mean_acc)
        mlflow.log_dict({'selected_features': selected}, 'selected_features.json')
        mlflow.log_input(mlflow_dataset, context="training")

    return mean_acc

# ===== RUN THE SEARCH (wrapped in one parent MLflow run) =====
with mlflow.start_run(run_name="optuna_search_parent") as parent_run:
    mlflow.log_input(mlflow_dataset, context="training")

    study = optuna.create_study(
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
    )
    study.optimize(objective, n_trials=100, show_progress_bar=True)

    # Pruned vs completed
    pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
    completed = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    print(f"Completed: {completed}, Pruned early: {pruned}")
    mlflow.log_metric('n_trials_completed', completed)
    mlflow.log_metric('n_trials_pruned', pruned)

    # ===== RESULTS =====

    # 1. Best accuracy
    print(f"Best CV Accuracy: {study.best_value:.4f}")
    mlflow.log_metric('best_cv_accuracy', study.best_value)

    # 2. Best feature combination
    best_features = [f for f in feature_names if study.best_params[f]]
    print(f"\nSelected Features ({len(best_features)}):")
    print(best_features)
    mlflow.log_dict({'best_features': best_features}, 'best_features.json')
    mlflow.log_metric('n_best_features', len(best_features))

    # 3. Best hyperparameters
    hyperparam_names = ['n_estimators', 'max_depth', 'min_samples_split']
    best_hyperparams = {k: v for k, v in study.best_params.items() if k in hyperparam_names}
    print(f"\nBest Hyperparameters:")
    print(best_hyperparams)
    mlflow.log_params({f"best_{k}": v for k, v in best_hyperparams.items()})

    # 4a. Feature importance
    feature_importance = optuna.importance.get_param_importances(study, params=feature_names)
    print(f"\nFeature Importance (weightage):")
    for param, score in feature_importance.items():
        print(f" {param}: {score:.4f}")
    mlflow.log_dict(feature_importance, 'feature_importance.json')

    fig1, ax1 = plt.subplots(figsize=(8, 10))
    ax1.barh(list(feature_importance.keys()), list(feature_importance.values()), color='steelblue')
    ax1.set_xlabel('Importance')
    ax1.set_title('Feature Importance (Optuna fANOVA)')
    ax1.invert_yaxis()
    plt.tight_layout()
    mlflow.log_figure(fig1, 'feature_importance.png')
    plt.show()

    # 4b. Hyperparameter importance
    hyperparam_importance = optuna.importance.get_param_importances(study, params=hyperparam_names)
    print(f"\nHyperparameter Importance (weightage):")
    for param, score in hyperparam_importance.items():
        print(f" {param}: {score:.4f}")
    mlflow.log_dict(hyperparam_importance, 'hyperparameter_importance.json')

    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.barh(list(hyperparam_importance.keys()), list(hyperparam_importance.values()), color='darkorange')
    ax2.set_xlabel('Importance')
    ax2.set_title('Hyperparameter Importance (Optuna fANOVA)')
    ax2.invert_yaxis()
    plt.tight_layout()
    mlflow.log_figure(fig2, 'hyperparameter_importance.png')
    plt.show()

    # 5. Train and log the final best model, so it's tied to this run
    final_model = RandomForestClassifier(**best_hyperparams, random_state=42)
    final_model.fit(X[best_features], y)
    mlflow.sklearn.log_model(final_model, "best_model")

    print(f"\nMLflow parent run ID: {parent_run.info.run_id}")